In [ ]:
# Install Tiny-SIMA and dependencies into the active Jupyter kernel.
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.60"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        "Tiny-SIMA requires a Python 3.11 or 3.12 Jupyter kernel."
    )

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

has_cuda = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    ).returncode == 0
)
extras = "notebook,cuda12" if has_cuda else "notebook"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])
print("Repository:", repo_root)
print("CUDA detected:", has_cuda)


# Profile executor scaling

This notebook measures PPO throughput at several environment batch sizes. Each candidate runs in a fresh Python process so JAX compilation and allocator state do not leak between measurements.

The candidates use the same number of PPO updates, not the same number of environment steps. This is a hardware-throughput experiment rather than a learning comparison.

In [ ]:
ENVIRONMENT_COUNTS = (64, 128, 256, 512, 1024, 2048, 4096)
NUM_UPDATES = 32
REUSE_EXISTING = True
output_dir = repo_root / "runs" / "executor-scaling"

print("Environment counts:", ENVIRONMENT_COUNTS)
print("Updates per candidate:", NUM_UPDATES)
print("Reuse completed candidates:", REUSE_EXISTING)
print("Output directory:", output_dir)


In [ ]:
command = [
    sys.executable,
    "-m",
    "tiny_sima.profiling",
    "--config",
    str(repo_root / "configs" / "ppo_1m.yaml"),
    "--updates",
    str(NUM_UPDATES),
    "--output-dir",
    str(output_dir),
    "--num-envs",
    *map(str, ENVIRONMENT_COUNTS),
]
if REUSE_EXISTING:
    command.append("--reuse-existing")
subprocess.check_call(command, cwd=repo_root)


In [ ]:
import json

with (output_dir / "scaling_summary.json").open() as file:
    results = json.load(file)

def formatted(value, format_spec):
    return format(value, format_spec) if value is not None else "n/a"

for result in results:
    print(
        f"envs={result['num_envs']:>3}  "
        f"tail_sps={formatted(result['tail_steps_per_second'], '10.0f')}  "
        "tail_gpu="
        f"{formatted(result['tail_mean_gpu_utilization_percent'], '5.1f')}%  "
        f"jax_peak={formatted(result['jax_peak_memory_mib'], '8.1f')} MiB"
    )
results


In [ ]:
import matplotlib.pyplot as plt

environment_counts = [result["num_envs"] for result in results]
throughput = [result["tail_steps_per_second"] for result in results]
utilization = [
    result["tail_mean_gpu_utilization_percent"] for result in results
]

figure, throughput_axis = plt.subplots(figsize=(8, 4.5))
throughput_axis.plot(environment_counts, throughput, marker="o")
throughput_axis.set_xlabel("Parallel environments")
throughput_axis.set_ylabel("Tail-window steps/second")
throughput_axis.grid(alpha=0.25)

utilization_axis = throughput_axis.twinx()
utilization_axis.plot(
    environment_counts,
    utilization,
    marker="s",
    color="tab:orange",
)
utilization_axis.set_ylabel("Tail-window GPU utilization (%)")
utilization_axis.set_ylim(0, 105)
plt.title("Tiny-SIMA executor scaling")
plt.show()
